[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/w05-w06-pca/lab/lab06_pca_practice.ipynb)

# Lab 06 — PCA II: PCA in practice

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, GridSearchCV
np.set_printoptions(precision=4, suppress=True)

**Course:** 040563107 Tools for Statistical Data Science · **Week 6** · **Time:** 3 hours · **Graded:** lab rubric (/8)

**After this lab you can:**
* choose the number of components with a scree plot, cumulative variance, the eigenvalue-above-one rule and Horn's parallel analysis (PCA-O5);
* fit PCA **inside** the validation boundary with nested cross-validation and compare it with a baseline (PCA-O6);
* measure how much leakage from a fitted step biases results, and explain why that depends on whether the step uses $y$ (PCA-O6);
* show that high-variance directions need not be predictive, and that a 2-D picture is not a predictive test (PCA-O7);
* quantify the influence of one outlier on PC1 (PCA-O8).

Work as in Lab 05: complete the **Tasks**, run the **Self-checks**, answer the **Q**s, then *Restart and run all* before submitting.

## Part 1 — How many components? (about 40 minutes)

We use correlation PCA of Wine (standardised variables). The given cell draws the scree plot and the cumulative explained variance.

In [ ]:
Xw, yw = tsds.data.load("wine")
Zw = StandardScaler().fit_transform(Xw)
eig_w = PCA().fit(Zw).explained_variance_
cum_w = np.cumsum(eig_w) / eig_w.sum()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(range(1, 14), eig_w, "o-"); ax[0].axhline(1, ls=":", c="grey")
ax[0].set(title="Scree plot (correlation PCA)", xlabel="component", ylabel="eigenvalue")
ax[1].plot(range(1, 14), cum_w, "o-"); ax[1].axhline(0.8, ls=":", c="grey")
ax[1].set(title="Cumulative explained variance", xlabel="k", ylabel="share")
plt.tight_layout(); plt.show()
print("eigenvalues:", eig_w.round(3))
print("eigenvalue > 1 rule keeps k =", int((eig_w > 1).sum()), "| first k reaching 80%:", int(np.argmax(cum_w >= 0.8)) + 1)

In [ ]:
# Task T1
def parallel_analysis(Z, B=200, q=95, seed=SEED):
    """Horn's parallel analysis for correlation PCA.

    For b = 1..B: permute EACH column of Z independently (this destroys correlation but keeps
    each variable's distribution), compute the eigenvalues of the correlation matrix of the
    permuted data, and sort them in descending order.
    Returns (thresholds, k):
      thresholds : (p,) array, the q-th percentile of the null eigenvalues at each rank
      k          : number of leading observed eigenvalues that exceed their threshold
                   (count from component 1 and stop at the first one that does not exceed)
    """
    Z = np.asarray(Z, dtype=float)
    gen = np.random.default_rng(seed)
    # YOUR CODE HERE
    raise NotImplementedError()

thr, k_pa = parallel_analysis(Zw)
print("95th-percentile null eigenvalues (first 5):", thr[:5].round(3), "| parallel analysis keeps k =", k_pa)

In [ ]:
# Self-check for task T1: run it; it should print OK.
assert thr.shape == (13,) and 1 <= k_pa <= 13
thr2, k2 = parallel_analysis(Zw)
assert np.allclose(thr, thr2) and k_pa == k2, "use the seeded generator `gen` so results are reproducible"
rng_check = np.random.default_rng(0)
_, k_noise = parallel_analysis(rng_check.normal(size=(178, 13)), B=50)
assert k_noise <= 1, "on pure noise parallel analysis should keep (almost) no components"
print("OK")

**Q1.** Compare the k suggested by the elbow of the scree plot, by 80% cumulative variance, by the eigenvalue-above-one rule, and by parallel analysis.
Which k would you report for (a) a 2-D figure in a report, and (b) "how many dimensions of systematic co-variation do these 13 measurements contain?"
Justify each in 2–3 sentences, and explain why "keep components until held-out reconstruction error stops falling" is not a valid rule.

*Your answer:* 

## Part 2 — Does PCA help prediction? Digits with nested cross-validation (about 50 minutes)

Digits: 1,797 images of 8×8 pixels (UCI 80, CC BY 4.0). All comparisons use the **same** 5-fold stratified split defined below. The given cell computes the majority-class baseline.

In [ ]:
Xd, yd = tsds.data.load("digits")
Xd, yd = Xd.to_numpy(), yd.to_numpy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
majority = cross_val_score(DummyClassifier(), Xd, yd, cv=cv)
print(Xd.shape, "| majority-class accuracy:", majority.mean().round(3))

In [ ]:
# Task T2
# (a) base_scores: 5-fold CV accuracies (use `cv`) of the baseline
#     make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)) on all 64 pixels.
# (b) nested: the output of cross_validate(..., cv=cv, return_estimator=True) for a GridSearchCV that
#     tunes pca__n_components over [5, 10, 20, 30, 40] with inner_cv, on the pipeline
#     make_pipeline(StandardScaler(), PCA(random_state=SEED), LogisticRegression(max_iter=5000)).
# (c) chosen_k: list of the k selected in each outer fold.
# YOUR CODE HERE
raise NotImplementedError()
print(f"baseline  : {base_scores.mean():.3f} +/- {base_scores.std():.3f}")
print(f"PCA nested: {nested['test_score'].mean():.3f} +/- {nested['test_score'].std():.3f} | k chosen per fold: {chosen_k}")

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert len(base_scores) == 5 and base_scores.mean() > 0.9
assert len(nested["test_score"]) == 5 and len(chosen_k) == 5
assert all(k in [5, 10, 20, 30, 40] for k in chosen_k)
assert all(getattr(est.cv, "random_state", None) == SEED + 1 for est in nested["estimator"]),     "tune k with inner_cv (a separate inner split), not with the outer split"
print("OK")

**Q2.** Does PCA improve accuracy here? Answer with the numbers and their fold-to-fold variability, and comment on the chosen k.
Name one *other* reason someone might still use PCA in this pipeline, and say what evidence would support that claim. (About 100 words.)

*Your answer:* 

## Part 3 — How large is leakage? (about 25 minutes)

Pure noise: $n=60$, $p=2000$, random labels, so the true accuracy is 0.5. `repeated_cv` averages 20 repetitions of 5-fold CV. Compare fitting a step on **all rows** (then CV) against fitting it **inside** the pipeline, for an unsupervised step (PCA) and a supervised step (SelectKBest, which uses $y$).

In [ ]:
gen = np.random.default_rng(SEED)
Xn = gen.normal(size=(60, 2000))
yn = gen.integers(0, 2, size=60)

def repeated_cv(model, X, y, n_rep=20):
    """Mean accuracy over n_rep repetitions of shuffled 5-fold stratified CV."""
    return float(np.mean([cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=r)).mean()
                          for r in range(n_rep)]))

In [ ]:
# Task T3
# Compute four numbers with repeated_cv and LogisticRegression(max_iter=5000) as the classifier:
#   pca_before : PCA(10, random_state=SEED) fitted on ALL rows of Xn first, then CV of the classifier on the transformed data
#   pca_inside : CV of make_pipeline(PCA(10, random_state=SEED), classifier)
# (random_state matters: for wide data sklearn switches PCA to a randomised solver)
#   skb_before : SelectKBest(f_classif, k=10) fitted on ALL rows (with yn) first, then CV of the classifier
#   skb_inside : CV of make_pipeline(SelectKBest(f_classif, k=10), classifier)
# YOUR CODE HERE
raise NotImplementedError()
print(pd.DataFrame({"fitted on all rows first": [pca_before, skb_before],
                    "inside the pipeline": [pca_inside, skb_inside]},
                   index=["PCA(10)  (ignores y)", "SelectKBest(10) (uses y)"]).round(3))

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert skb_before > skb_inside + 0.2, "supervised selection before the split should look much better than it is"
assert abs(skb_inside - 0.5) < 0.15 and abs(pca_inside - 0.5) < 0.15, "inside the pipeline, noise should score near 0.5"
print("OK")

## Part 4 — Variance is not relevance; a picture is not a test (about 30 minutes)

The class information sits in a **low-variance** direction. The given cell generates and plots the data.

In [ ]:
gen = np.random.default_rng(SEED)
n_syn = 400
y_syn = gen.integers(0, 2, n_syn)
x1 = gen.normal(0, 5, n_syn)                                  # large variance, no class information
x2 = gen.normal(0, 0.5, n_syn) + 1.2 * (2 * y_syn - 1)       # small variance, all of the class information
X_syn = np.column_stack([x1, x2])
print("sample variances:", X_syn.var(axis=0, ddof=1).round(2))
plt.figure(figsize=(5, 3.5)); plt.scatter(x1, x2, c=y_syn, s=8, cmap="coolwarm")
plt.xlabel("x1 (variance about 26.6)"); plt.ylabel("x2 (variance about 1.7)"); plt.title("Synthetic two-class data"); plt.show()

In [ ]:
# Task T4
# Fill the dict `acc` with 5-fold CV accuracies (use `cv`) of
#   "pca1": make_pipeline(PCA(1), LogisticRegression())
#   "lda1": make_pipeline(LinearDiscriminantAnalysis(n_components=1), LogisticRegression())
#   "raw" : LogisticRegression() on both variables
# YOUR CODE HERE
raise NotImplementedError()
{k: round(v, 3) for k, v in acc.items()}

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert set(acc) == {"pca1", "lda1", "raw"}
assert acc["lda1"] > acc["pca1"] + 0.3
print("OK")

In [ ]:
# Digits: what a 2-D PCA picture shows, compared with what a classifier can use.
from sklearn.neighbors import KNeighborsClassifier
Z2 = PCA(2, random_state=SEED).fit_transform(StandardScaler().fit_transform(Xd))   # for display only
plt.figure(figsize=(5, 4)); plt.scatter(Z2[:, 0], Z2[:, 1], c=yd, s=6, cmap="tab10"); plt.colorbar(label="digit")
plt.title("Digits: first two principal components"); plt.show()
for k in (2, 30):
    s = cross_val_score(make_pipeline(StandardScaler(), PCA(k, random_state=SEED), KNeighborsClassifier()), Xd, yd, cv=cv)
    print(f"k-NN on {k:>2} PCs: {s.mean():.3f}")

**Q3.** Using your `acc` values and the digits output above, explain (a) why PCA(1) fails on the synthetic data while LDA(1) succeeds, and
(b) why "the classes overlap in the 2-D PCA plot, so the digits are hard to classify" is a wrong inference. (About 120 words.)

*Your answer:* 

## Part 5 — One outlier (about 20 minutes)

Measure how much **one** contaminated observation of 178 rotates PC1 of standardised Wine.

In [ ]:
# Task T5
def angle_deg(u, v):
    """Angle in degrees between the LINES spanned by u and v (so the sign of a component does not matter)."""
    # YOUR CODE HERE
    raise NotImplementedError()

pc = PCA(2).fit(Zw)
Z_out = Zw.copy()
Z_out[0] = Z_out[0] + 15 * pc.components_[1]       # push observation 0 far along the PC2 direction
pc1_clean, pc1_out = pc.components_[0], PCA(1).fit(Z_out).components_[0]
outlier_angle = angle_deg(pc1_clean, pc1_out)
print(f"PC1 rotates by {outlier_angle:.1f} degrees because of one point")

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert np.isclose(angle_deg([1, 0], [0, 1]), 90) and np.isclose(angle_deg([1, 0], [-1, 0]), 0)
assert outlier_angle > 5
print("OK")

**Q4.** **Failure analysis and interpretation.** Name two ways a PCA analysis in this lab could have misled you (choose from: k chosen on the test data, a supervised step fitted before the split, reading a 2-D plot as a predictive test, an undiagnosed outlier). For each, give the evidence from this lab and the safeguard you would use in your project. (About 150 words.)

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| | | | |

If you used no AI tool, write "None".